## NeuroMTA Framework Example
### Example 4: Inter-core Communication

In [1]:
import math
import torch

from neuromta.framework import *

In [2]:
class CoreMap:
    def __init__(
        self, 
        
        npu_core_num: int,
        npu_core_l1_bank_size: int,
        
        dma_core_num: int,
        dma_core_mem_bank_size: int, 
    ):
        self._mem_info: dict[str, tuple[int, int]] = {   # mem_type -> (bank_size, num_banks)
            "NPU_L1":  (npu_core_l1_bank_size,  npu_core_num),
            "DMA_MEM": (dma_core_mem_bank_size, dma_core_num),
        }
        
        self._mem_to_core_id_map: dict[tuple[str, int], int] = {}
        self._core_id_to_mem_map: dict[int, tuple[str, int]] = {}
    
    def add_core(self, mem_type: str, bank_id: int) -> int:
        core_id = len(self._core_id_to_mem_map)
        
        mem_signature = (mem_type, bank_id)
        
        self._mem_to_core_id_map[mem_signature] = core_id
        self._core_id_to_mem_map[core_id] = mem_signature
        
        return core_id
    
    def get_mem_info(self, core_id: int) -> tuple[int, int]:
        mem_type, bank_id = self._core_id_to_mem_map[core_id]
        bank_size, _ = self._mem_info[mem_type]
        base_addr = bank_id * bank_size
        return base_addr, bank_size

    def get_mem_type(self, core_id: int) -> str:
        mem_type, _ = self._core_id_to_mem_map[core_id]
        return mem_type
    
    def get_mem_owner(self, addr: int) -> int:
        for (mem_type, bank_id), core_id in self._mem_to_core_id_map.items():
            bank_size, _ = self._mem_info[mem_type]
            base_addr = bank_id * bank_size
            if base_addr <= addr < base_addr + bank_size:
                return core_id
        raise ValueError(f"No core owns the address {addr:#x}")

In [11]:
class SimpleNPUCore(Core):
    def __init__(self, core_map: CoreMap, core_id: int):
        super().__init__(core_id, SimpleNPUCoreCycleModel())
        
        self.core_map = core_map

        l1_base_addr, l1_size = self.core_map.get_mem_info(core_id)
        
        self.l1_memory = MemoryHandle(
            mem_id=core_id,
            base_addr=l1_base_addr,
            size=l1_size,
        )
        
        self.mxu_pe_arr = torch.zeros((128, 128), dtype=torch.int32)
        
        self.mxu_lock      = self.l1_memory.allocate_var_ptr(4, initial_value=0)
        self.l1_read_lock  = self.l1_memory.allocate_var_ptr(4, initial_value=0)
        self.l1_write_lock = self.l1_memory.allocate_var_ptr(4, initial_value=0)
    
    @core_command_method
    def mxu_compute(
        self, 
        
        ifm: DataContainer[torch.Tensor], 
        wgt: DataContainer[torch.Tensor], 
        psum: DataContainer[torch.Tensor], 
        ofm: DataContainer[torch.Tensor],
    
        preload_psum: bool = True,
        flush_ofm: bool = True,
    ):
        if preload_psum:
            psum.data = psum.data.view(torch.int32).reshape(128, 128)
            self.mxu_pe_arr[:, :] = psum.data

        ifm.data = ifm.data.view(torch.int32).reshape(128, 128)
        wgt.data = wgt.data.view(torch.int32).reshape(128, 128)
        
        self.mxu_pe_arr[:, :] = torch.matmul(ifm.data, wgt.data) + self.mxu_pe_arr

        if flush_ofm:
            ofm.data = self.mxu_pe_arr.clone()
            self.mxu_pe_arr[:, :] = 0
            
    def read_single_page(self, ptr: BufferPointer, container: DataContainer[torch.Tensor]):
        owner_id = None
        for page_ptr in ptr.raw_handle.page_ptrs:
            if owner_id is None:
                owner_id = self.core_map.get_mem_owner(page_ptr.addr)
            else:
                assert owner_id == self.core_map.get_mem_owner(page_ptr.addr), "All pages must be in the same memory region"
                
        if owner_id == self.core_id:
            self.l1_read_single_page(ptr, container)
        else:
            msg = RPCMessage(
                src_core_id=self.core_id,
                dst_core_id=owner_id,
                cmd_id="dram_read" if self.core_map.get_mem_type(owner_id) == "DMA_MEM" else "l1_read_single_page",
            ).with_args(ptr, container)
            
            self.async_rpc_send_req_msg(msg)
            self.async_rpc_wait_rsp_msg(msg)
            
    def write_single_page(self, ptr: BufferPointer, container: DataContainer[torch.Tensor]):
        owner_id = None
        for page_ptr in ptr.raw_handle.page_ptrs:
            if owner_id is None:
                owner_id = self.core_map.get_mem_owner(page_ptr.addr)
            else:
                assert owner_id == self.core_map.get_mem_owner(page_ptr.addr), "All pages must be in the same memory region"

        if owner_id == self.core_id:
            self.l1_write_single_page(ptr, container)
        else:
            msg = RPCMessage(
                src_core_id=self.core_id,
                dst_core_id=owner_id,
                cmd_id="dram_write" if self.core_map.get_mem_type(owner_id) == "DMA_MEM" else "l1_write_single_page",
            ).with_args(ptr, container)
            
            self.async_rpc_send_req_msg(msg)
            self.async_rpc_wait_rsp_msg(msg)
        
    @core_command_method
    def l1_read_single_page(self, ptr: BufferPointer, container: DataContainer[torch.Tensor]):
        container.data = self.l1_memory.get_content(ptr, shape=(128, 128), dtype=torch.int32)

    @core_command_method
    def l1_write_single_page(self, ptr: BufferPointer, container: DataContainer[torch.Tensor]):
        self.l1_memory.set_content(ptr, container.data)
        
    @core_conditional_command_method
    def lock_wait(self, lock_ptr: Pointer, value: int):
        return self.l1_memory.get_content(lock_ptr) == value
    
    @core_command_method
    def lock_atomic_inc(self, lock_ptr: Pointer):
        current_value = self.l1_memory.get_content(lock_ptr)
        self.l1_memory.set_content(lock_ptr, current_value + 1)

class SimpleNPUCoreCycleModel(CoreCycleModel):
    def __init__(self):
        super().__init__()
        
    def mxu_compute(
        self,
        ifm: DataContainer[torch.Tensor],
        wgt: DataContainer[torch.Tensor],
        psum: DataContainer[torch.Tensor],
        ofm: DataContainer[torch.Tensor],
        preload_psum: bool = True,
        flush_ofm: bool = True,
    ):
        i = 1
        if preload_psum: i += 1
        if flush_ofm: i += 1
        return 128 * i 
    
    def l1_read_single_page(self, ptr: BufferPointer, container: DataContainer[torch.Tensor]):
        return 64
    
    def l1_write_single_page(self, ptr: BufferPointer, container: DataContainer[torch.Tensor]):
        return 64

In [12]:
class SimpleDMACore(Core):
    def __init__(self, core_map: CoreMap, core_id: int):
        super().__init__(core_id, SimpleDMACoreCycleMode())
        
        self.core_map = core_map
        
        self.dram_memory = MemoryHandle(
            mem_id="DRAM",
            base_addr=0x80000000,
            size=parse_mem_cap_str("1GB")
        )
        
    @core_command_method
    def dram_read(self, ptr: BufferPointer, container: DataContainer[torch.Tensor]):
        container.data = self.dram_memory.get_content(ptr)
        
    @core_command_method
    def dram_write(self, ptr: BufferPointer, container: DataContainer[torch.Tensor]):
        self.dram_memory.set_content(ptr, container.data)

class SimpleDMACoreCycleMode(CoreCycleModel):
    def __init__(self):
        super().__init__()
        
        self.access_gran = 64  # bytes per cycle
        self.latency = 400     # fixed latency in cycles
        
    def dram_read(self, ptr: BufferPointer, container: DataContainer[torch.Tensor]):
        return math.ceil(ptr.page_size * ptr.n_pages / self.access_gran) + self.latency

    def dram_write(self, ptr: BufferPointer, container: DataContainer[torch.Tensor]):
        return math.ceil(ptr.page_size * ptr.n_pages / self.access_gran) + self.latency

In [13]:
class SimpleNPUDevice(Device):
    def __init__(self, n_npu_cores: int, n_dma_cores: int):
        super().__init__()
        
        self.n_npu_cores = n_npu_cores
        self.n_dma_cores = n_dma_cores
        
        self.core_map = CoreMap(
            npu_core_num=n_npu_cores,
            npu_core_l1_bank_size=parse_mem_cap_str("2MB"),
            dma_core_num=n_dma_cores,
            dma_core_mem_bank_size=parse_mem_cap_str("4GB"),
        )
        
        self.npu_cores = [
            SimpleNPUCore(core_map=self.core_map, core_id=self.core_map.add_core("NPU_L1", bank_id=i))
            for i in range(n_npu_cores)
        ]
        
        self.dma_cores = [
            SimpleDMACore(core_map=self.core_map, core_id=self.core_map.add_core("DMA_MEM", bank_id=i))
            for i in range(n_dma_cores)
        ]

In [14]:
@jit_prototype
def example_kernel(
    core: SimpleNPUCore, 
    
    ifm: BufferPointer,
    wgt: BufferPointer,
    psum: BufferPointer,
    ofm: BufferPointer,
):
    containers = [DataContainer() for _ in range(4)]
    
    core.read_single_page(ifm, containers[0])    # read a single page no matter the pointer refers to the l1 or dram address space
    core.read_single_page(wgt, containers[1])    # read a single page no matter the pointer refers to the l1 or dram address space
    core.read_single_page(psum, containers[2])   # read a single page no matter the pointer refers to the l1 or dram address space

    core.mxu_compute(*containers)

    core.write_single_page(ofm, containers[3])   # write a single page no matter the pointer refers to the l1 or dram address space

In [15]:
device = SimpleNPUDevice(n_npu_cores=1, n_dma_cores=1)  # Initialize device with 1 core
device.initialize()

In [16]:
logger.set_print_options(log_level=LogLevel.DEBUG)
device.set_command_debug_verbosity(verbose=True)

In [17]:
dma_core = device.dma_cores[0]

ifm  = dma_core.dram_memory.allocate_buffer_ptr(page_size=128*128*4, n_pages=1)
wgt  = dma_core.dram_memory.allocate_buffer_ptr(page_size=128*128*4, n_pages=1)
psum = dma_core.dram_memory.allocate_buffer_ptr(page_size=128*128*4, n_pages=1)
ofm  = dma_core.dram_memory.allocate_buffer_ptr(page_size=128*128*4, n_pages=1)

ifm_tensor  = torch.randint(0, 10, (128, 128), dtype=torch.int32)
wgt_tensor  = torch.randint(0, 10, (128, 128), dtype=torch.int32)
psum_tensor = torch.randint(0, 10, (128, 128), dtype=torch.int32)

dma_core.dram_memory.set_content(ifm, ifm_tensor)
dma_core.dram_memory.set_content(wgt, wgt_tensor)
dma_core.dram_memory.set_content(psum, psum_tensor)

In [18]:
npu_core = device.npu_cores[0]

kernel = example_kernel(npu_core, ifm, wgt, psum, ofm)
kernel.dispatch(slot_id="main")

device.run_kernels()

[DEBUG] 0      - 1      | 0          | MAIN<main>::example_kernel                                                                           | command: async_rpc_send_req_msg
[DEBUG] 1      - 2      | 0          | MAIN<main>::example_kernel                                                                           | command: async_rpc_wait_rsp_msg
[DEBUG] 1      - 1425   | 1          | RPC<0>::AUTO_REMOTE.0                                                                                | command: dram_read
[DEBUG] 1425   - 1426   | 0          | MAIN<main>::example_kernel                                                                           | command: async_rpc_send_req_msg
[DEBUG] 1426   - 1427   | 0          | MAIN<main>::example_kernel                                                                           | command: async_rpc_wait_rsp_msg
[DEBUG] 1426   - 2850   | 1          | RPC<0>::AUTO_REMOTE.0                                                                                | c

In [19]:
print(f"simulation terminated in {device.timestamp} cycles")
    
reference = torch.matmul(ifm_tensor, wgt_tensor) + psum_tensor
simulated = dma_core.dram_memory.get_content(ofm, shape=(128, 128), dtype=torch.int32)
print(f"simulation {'PASSED' if torch.equal(reference, simulated) else 'FAILED'}")
print("\nReference Output:\n", reference)
print("\nSimulated Output:\n", simulated)

simulation terminated in 6085 cycles
simulation PASSED

Reference Output:
 tensor([[2415, 2661, 2535,  ..., 2809, 2592, 2639],
        [2628, 2856, 2694,  ..., 2957, 2694, 2507],
        [2864, 2883, 2622,  ..., 3054, 2710, 2612],
        ...,
        [2439, 2742, 2683,  ..., 2845, 2540, 2679],
        [2395, 2704, 2597,  ..., 2660, 2525, 2411],
        [2481, 2726, 2642,  ..., 2936, 2583, 2462]], dtype=torch.int32)

Simulated Output:
 tensor([[2415, 2661, 2535,  ..., 2809, 2592, 2639],
        [2628, 2856, 2694,  ..., 2957, 2694, 2507],
        [2864, 2883, 2622,  ..., 3054, 2710, 2612],
        ...,
        [2439, 2742, 2683,  ..., 2845, 2540, 2679],
        [2395, 2704, 2597,  ..., 2660, 2525, 2411],
        [2481, 2726, 2642,  ..., 2936, 2583, 2462]], dtype=torch.int32)
